# Japanese Handwriting Recognition — Project Journey & Findings

This notebook is a retrospective companion to the project: it re-runs, in one place, the key discoveries, bugs, and decisions that shaped `src/data_loader.py`, `src/train.py`, `src/evaluate.py`, and `src/predictor.py` during development. It isn't part of the production pipeline, but it's here so anyone looking at this repo can retrace the reasoning, not just the final code.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # combine_datasets() uses paths relative to the project root

import sys
sys.path.append("src")

import numpy as np
import torch
from PIL import Image

print("Working directory:", os.getcwd())

Working directory: /Users/kursad/Desktop/Projekte/Japanese Handwriting Recognition


## 1. Kuzushiji-Kanji: an extremely imbalanced dataset

The full Kuzushiji-Kanji dataset has 3,832 classes but it's wildly imbalanced. When first inspecting it, **815 of the 3,832 classes had exactly 1 image**, and the median class had only 6 images. Classes that thin are unusable for training a CNN.

The fix: keep only the 49 classes with the *most* images, so the kanji half of the project has roughly the same number of classes as the 46 hiragana characters (46 + 49 = 95 total classes). We then deleted the other 3,783 folders on disk, so that exact distribution can't be re-derived here anymore but the current, pruned state is fully reproducible:

In [2]:
kanji_root = "data/kuzushiji_kanji/kkanji2"
kanji_counts = {
    d: len(os.listdir(os.path.join(kanji_root, d)))
    for d in os.listdir(kanji_root)
    if os.path.isdir(os.path.join(kanji_root, d))
}

print("Classes kept:", len(kanji_counts))
print("Images per class: min =", min(kanji_counts.values()), " max =", max(kanji_counts.values()))
print("Total kanji images:", sum(kanji_counts.values()))

Classes kept: 49
Images per class: min = 491  max = 1768
Total kanji images: 40371


491 to 1,768 images per class is a solid amount to train on, comfortably more than the 100 images/class on the hiragana side, which turned out to matter later (see Section 4).

## 2. Two datasets, two opposite color conventions

The Kaggle hiragana images and the Kuzushiji-Kanji images come from completely different sources, and it turns out they use **opposite pixel conventions**: one is a photo-like white background with dark ink, the other already follows the MNIST-style dark background/light stroke convention. Mixing them without fixing this would have taught the CNN two contradictory definitions of "background".

In [3]:
hira_folder = "data/kaggle_hiragana/aa"
hira_file = os.listdir(hira_folder)[0]
hira_img = np.array(Image.open(os.path.join(hira_folder, hira_file)).convert("L"))

kanji_folder = "data/kuzushiji_kanji/kkanji2/U+4E00"
kanji_file = os.listdir(kanji_folder)[0]
kanji_img = np.array(Image.open(os.path.join(kanji_folder, kanji_file)).convert("L"))

print(f"Hiragana raw mean pixel value: {hira_img.mean():.1f}  (bright -> white background)")
print(f"Kanji raw mean pixel value:    {kanji_img.mean():.1f}  (dark -> black background)")

Hiragana raw mean pixel value: 236.7  (bright -> white background)
Kanji raw mean pixel value:    28.2  (dark -> black background)


That's why `load_hiragana()` inverts every pixel (`255 - pixel`) after converting to grayscale, while `load_kanji()` doesn't invert at all. The two loaders look almost identical, but that one line encodes a real, verified difference between the sources.

## 3. The seed bug — accidental train/test leakage

`combine_datasets()` shuffles all 44,971 images with `np.random.permutation()` before splitting into train/test. The first version of this function had **no fixed random seed**, so every call produced a *different* shuffle:

In [4]:
order_1 = np.random.permutation(10)
order_2 = np.random.permutation(10)
print("Call 1:", order_1)
print("Call 2:", order_2)
print("Identical?", (order_1 == order_2).all())

Call 1: [3 6 8 7 9 0 5 2 1 4]
Call 2: [8 1 2 0 9 5 7 4 3 6]
Identical? False


That's a real problem: `train.py` calls `combine_datasets()` once to train the model, and `evaluate.py` calls it *again* to rebuild the test set for evaluation. Without a fixed seed, these two calls produce **different** splits, so `evaluate.py` was accidentally testing on a mix that included many images the model had already trained on. The symptom: a suspicious **99.64%** "test" accuracy, far above the ~97% the training run itself reported.

The fix was a single line: `np.random.seed(42)` right before the permutation in `combine_datasets()`. Now every call produces the exact same split:

In [5]:
from data_loader import combine_datasets

_, y_call_1, _ = combine_datasets()
_, y_call_2, _ = combine_datasets()
print("Two combine_datasets() calls now identical?", (y_call_1 == y_call_2).all())

Two combine_datasets() calls now identical? True


After the fix, the model was retrained and re-evaluated on a test set that's now guaranteed to be the same one used during training's own held-out split — and reproducible here too:

In [6]:
from model import CNN
from data_loader import prepare_data, create_dataloaders
from evaluate import get_predictions

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

state_dict = torch.load("models/cnn_handwriting_model.pth", map_location="cpu")
model = CNN().to(device)
model.load_state_dict(state_dict)

combined_X, combined_y, all_classes = combine_datasets()
X_train, y_train, X_test, y_test = prepare_data(combined_X, combined_y)
train_loader, test_loader = create_dataloaders(X_train, y_train, X_test, y_test)

predictions, labels = get_predictions(model, test_loader, device)
print(f"Real, leakage-free test accuracy: {(predictions == labels).mean():.2%}")

Real, leakage-free test accuracy: 97.71%


**97.71%**, a believable number, well clear of the inflated 99.64%, and consistent with what `train.py` itself reported during training.

## 4. The suspiciously perfect hiragana accuracy

Once the leakage bug was fixed, the per-class `classification_report` showed something else worth double-checking: **all 46 hiragana classes scored a perfect 1.00 precision/recall/F1**, while kanji classes ranged 0.91-1.00. A perfect score across *every single class* is exactly the kind of result that deserves scrutiny before celebrating it.

In [7]:
from sklearn.metrics import classification_report

report = classification_report(labels, predictions, target_names=all_classes, output_dict=True)
hiragana_f1 = [report[name]["f1-score"] for name in all_classes[:46]]
kanji_f1 = [report[name]["f1-score"] for name in all_classes[46:]]

print(f"Hiragana F1 range: {min(hiragana_f1):.2f} - {max(hiragana_f1):.2f}")
print(f"Kanji F1 range:    {min(kanji_f1):.2f} - {max(kanji_f1):.2f}")

Hiragana F1 range: 1.00 - 1.00
Kanji F1 range:    0.92 - 1.00


First suspicion: duplicate images between train and test. Checked directly and found only 4 near-duplicates out of 4,600 hiragana images, nowhere near enough to explain a perfect score on every class:

In [8]:
from data_loader import load_hiragana

X_hira, y_hira, hira_classes = load_hiragana("data/kaggle_hiragana")
flat = X_hira.reshape(len(X_hira), -1)
unique_rows = np.unique(flat, axis=0)
print(f"Hiragana images: {len(X_hira)} total, {len(unique_rows)} unique, {len(X_hira) - len(unique_rows)} duplicates")

Hiragana images: 4600 total, 4596 unique, 4 duplicates


The real explanation: the Kaggle hiragana images have much **lower intra-class variance** than the Kuzushiji-Kanji images. They look like they were generated from a small number of templates with light synthetic variation, rather than sampled from hundreds of independent historical writers the way the kanji were. That makes the 46-way hiragana classification task genuinely easier. Not a leak, just an asymmetry between the two data sources.

## 5. Adaptive polarity detection in the predictor

`predictor.py` has to handle images from *either* convention (Section 2) when someone feeds it a brand-new image. Instead of assuming a fixed convention, `preprocess_img()` decides per-image: if the mean pixel value is bright (> 127), the background is assumed to be light and the image gets inverted; otherwise it's left alone. This works because in any single-character image, the background covers most of the pixels.

In [9]:
for name, raw_img in [("Hiragana", hira_img), ("Kanji", kanji_img)]:
    resized = np.array(Image.fromarray(raw_img).resize((28, 28)))
    decision = "invert" if resized.mean() > 127 else "leave as-is"
    print(f"{name}: mean = {resized.mean():.1f} -> {decision}")

Hiragana: mean = 235.8 -> invert
Kanji: mean = 28.4 -> leave as-is


Both known conventions get classified correctly by this single rule, without hard-coding which dataset an image came from. So a genuinely new photo (of either polarity) should be handled correctly too.

## 6. A second leakage bug — this time in the demo script

The first version of `predictor.py`'s sample-prediction demo picked 10 random files directly from the raw `data/kaggle_hiragana/` and `data/kuzushiji_kanji/kkanji2/` folders. That's the same mistake as Section 3, just in a different place: those folders contain **both** the training and test images mixed together, with no on-disk separation. A randomly chosen file had roughly an 80% chance of being an image the model had already trained on, making the "proof" of good predictions look better than it honestly was.

The fix: draw the 10 samples from `X_test`/`y_test` (the same reproducible split used everywhere else in evaluation), not from the raw folders:

In [10]:
import random
from evaluate import class_to_char
from predictor import predict

sample_idx = random.choice(range(len(X_test)))
sample_tensor = torch.tensor(X_test[sample_idx], dtype=torch.float32).unsqueeze(0)
predicted_class, confidence = predict(model, sample_tensor, all_classes, device)

print("True label:     ", class_to_char(all_classes[y_test[sample_idx]]))
print("Predicted:      ", class_to_char(predicted_class))
print(f"Confidence:      {confidence:.1%}")
print("\n(this sample is guaranteed to come from the held-out test split, never seen during training)")

True label:      是
Predicted:       是
Confidence:      100.0%

(this sample is guaranteed to come from the held-out test split, never seen during training)


After this fix, `results/sample_predictions.png` occasionally shows a genuine mistake (e.g. a true 日 predicted as 物 at 97.8% confidence) instead of a suspicious 10-for-10, which is actually a *more* trustworthy result, since it matches the real ~97.7% test accuracy instead of silently benefiting from leakage.

## Known limitations

- **Kanji coverage is tiny relative to the real writing system**: only the 49 best-represented classes out of 3,832 possible Kuzushiji-Kanji characters are supported. Most kanji simply didn't have enough training images to be usable (Section 1).
- **The train/test split is not stratified per class**: `prepare_data()` splits the whole shuffled 44,971-image array 80/20, so individual classes end up with slightly different test-set sizes by chance (hiragana classes range from 12 to 29 test images, expected value 20). The overall 80/20 ratio is correct; the per-class ratio isn't guaranteed.
- **Hiragana accuracy is not directly comparable to kanji accuracy**: the perfect hiragana scores reflect a low-variance source dataset, not necessarily how well the model would generalize to more diverse real-world hiragana handwriting (Section 4).
- **The polarity-detection heuristic (Section 5) assumes the background covers most of the image.** An unusual input where ink/strokes cover more than half the frame could be inverted incorrectly.